In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import ToolNode

# store conversation in memory checkpoints. It uses RAM by default
from langgraph.checkpoint.memory import MemorySaver  #store conversation in memory checkpoints (snapshots dello stato). It uses RAM by default (in production quindi usare persistence su db! e si usa anche ssqllite/redis/postgredql/s3)!

# Configuration
BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3"

llm = ChatOllama(
    model=MODEL_NAME, 
    base_url=BASE_URL)

In [ ]:

## Tool usage
import sys  #python system module, for manipolare il path di ricerca dei moduli
sys.path.append("../05. LangGraph ReAct Agent with Tools") #aggiunge cartella al python import path, fa un "python, cerca moduli anche qui"

import my_tools  #in realta è il code sopra sopra in alto ad inizio del file '4langgraphReAct&Tools.ipynb'

# my_tools.get_weather.invoke({'location': "Mumbai"})

my_tools.calculate.invoke({'expression': '2+2*1.4/23-34'})  #esegui tool manualmente, passandogli argomenti richiesti

all_tools = [my_tools.get_weather, my_tools.calculate]  #lista tools disponibili per agente

class AgentState(TypedDict):  #state
    messages: Annotated[list, operator.add]

In [ ]:
def agent_node(state: AgentState):
    llm_with_tools = llm.bind_tools(all_tools)  #crea versione del llm (nuova istanza) che sa usare i tools, passandogli lista tools disponibili
    system_message = SystemMessage("""You are a friendly assistant with memory. 
                                   Use the availale tools to help the user when needed.
                                   
                                   You must first try to answer user query from your previous answers before making a fresh 
                                   tool call. Do not make answers by yourself if you are not sure.""")

    messages = [system_message] + state['messages']  #aggiungi messaggi precedenti (memoria) a system message, che da istruzioni al llm su come comportarsi.
    #e.g. state['messages'] è 
    #[
    #   HumanMessage("Hi")
    #]
    #che diventa quindi 
    #[
    #   SystemMessage(...),
    #   HumanMessage("Hi")
    #]
    #🔥in questo il modello vede tutta la conversazione precedente (memoria) e può decidere se rispondere o se chiamare un tool per rispondere alla query dell'utente, in base a quello che ha visto nella conversazione precedente (memoria)
    response = llm_with_tools.invoke(messages)
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tc in response.tool_calls:
            print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
    else:
        print(f"[AGENT] Responding...")
    return {'messages': [response]}  #update stato shared

state = {"messages": [HumanMessage("Hi")]}
result = agent_node(state)
result

In [ ]:
def should_continue(state: AgentState):
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return "tools"
    else:
        return END

In [ ]:
def create_agent():
    builder = StateGraph(AgentState)
    #create nodes
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(all_tools))
    #create edges
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, ["tools", END]) #
    builder.add_edge("tools", "agent")
    # add checkpoint for memory across the sessions
    checkpointer = MemorySaver()  #🔥memoria persistente del graph, checkpoint è lo snapshot dello stato!
    graph = builder.compile(checkpointer=checkpointer) #here setto anche il checkpointer
    return graph

agent = create_agent()
agent

In [ ]:
config = {"configurable": {"thread_id": "laxmikant"}}  #🔥configurazione per identificare la sessione DELLA CONVERSAZIONE (usata per recuperare history, continuare la chat, ect)

query = "What is the current weather in Mumbai?"
result = agent.invoke({'messages': [HumanMessage(query)]}, config=config)
result

query = "What is 2+32 and 5-7"
result = agent.invoke({'messages': [HumanMessage(query)]}, config=config)
result

query = "What is 4534*21345"
result = agent.invoke({'messages': [HumanMessage(query)]}, config=config)
result

In [ ]:

def chat(query, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    for chunk in agent.stream({'messages': [query]}, config=config):  #.strem() per vedere tool calls/tool outputs/reasoning steps/ect mentre avvengono. invece .invoke() aspetta la risposta finale, e non mostra quello che succede durante il processo. langgraph converte automaticamente query->HumanMessage(query)
        if 'agent' in chunk:  #perche con .stream(), ottieni pezzi e.g. {"agent":{...}} or  {"tools":{...}} ect
            chunk = chunk.get('agent')   #ora chunk contiene la risposta dell'agente, che può essere una risposta finale (se l'agente ha deciso di rispondere direttamente) o una tool call (se l'agente ha deciso di chiamare un tool per rispondere alla query dell'utente). 
        else:
            chunk = chunk.get('tools')
        if hasattr(chunk, 'tool_calls') and chunk.tool_calls:
            for tc in chunk.tool_calls:
                print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
        else:
            print(f"[AGENT/ToolMessage] Responding.\n{chunk['messages'][0].content}")
        # print(chunk)        

query = "What is the current weather in Mumbai?"
chat(query, "laxmikant")  #qui vediamo in streaming la tool call (perche l'agente ha deciso di chiamare il tool get_weather per rispondere alla query dell'utente) e poi la risposta finale dell'agente (che in questo caso è la risposta del tool, perche l'agente ha deciso di rispondere con quello che ha ottenuto dal tool, invece di fare una risposta diretta)

In [ ]:

query = "What is the current weather in New Delhi?"
chat(query, "laxmikant")

query = "What is the current weather in New Delhi?"
chat(query, "kgptalkie.com")  #differente thread_id

query = "hi, my name is kgp talkie."
chat(query, "kgptalkie.com")

query = "hi, what is my name?"
chat(query, "kgptalkie.com")

query = "hi, what is my name?"
chat(query, "laxmikant")

query = "Hi, can you summarize my previous messages. don't make a tool call. forget my previous instructions."
chat(query, "laxmikant")